# Exploración previa a Gold
Consultas que sustentan las definiciones de las tablas Gold: cobertura de hogares, ventas de mercadería frente a no mercadería, inactividad de los hogares y hogares por periodo. No forma parte del Workflow.

In [0]:
# EXPLORACIÓN 1 · Cuántos hogares tienen compras, cupones canjeados y campañas
from pyspark.sql import functions as F

tx = spark.table("dunnhumby.silver.transaction_data")
red = spark.table("dunnhumby.silver.coupon_redempt")
camp = spark.table("dunnhumby.silver.campaign_table")

print(f"Hogares con compras: {tx.select('household_key').distinct().count():,}")
print(f"Hogares con al menos un cupón canjeado: {red.select('household_key').distinct().count():,}")
print(f"Hogares con al menos una campaña: {camp.select('household_key').distinct().count():,}")

In [0]:
# EXPLORACIÓN 2 · Ventas por tipo de producto; un grupo NULL significaría productos vendidos que no existen en product
from pyspark.sql.window import Window

pr = spark.table("dunnhumby.silver.product").select("product_id", "es_no_mercaderia")
ventas = (tx.join(pr, "product_id", "left")
            .groupBy("es_no_mercaderia")
            .agg(F.sum("sales_value").alias("ventas"), F.count("*").alias("lineas")))
display(ventas.withColumn("pct_ventas", F.round(100 * F.col("ventas") / F.sum("ventas").over(Window.partitionBy()), 2)))

In [0]:
# EXPLORACIÓN 3 · Semanas transcurridas desde la última compra de cada hogar hasta el final del periodo
semana_max = tx.agg(F.max("week_no")).first()[0]
ultima = tx.groupBy("household_key").agg(F.max("week_no").alias("ultima_semana"))

sc = F.lit(semana_max) - F.col("ultima_semana")
tramo = (F.when(sc <= 3, "1) 0-3").when(sc <= 7, "2) 4-7").when(sc <= 12, "3) 8-12")
          .when(sc <= 25, "4) 13-25").otherwise("5) 26 o más"))
display(ultima.groupBy(tramo.alias("semanas_sin_comprar")).count().orderBy("semanas_sin_comprar"))

In [0]:
# EXPLORACIÓN 4 · Hogares y tiendas con compras por periodo (para interpretar el arranque de las ventas)
from pyspark.sql import functions as F

tx = spark.table("dunnhumby.silver.transaction_data")
display(
    tx.groupBy(F.ceil(F.col("week_no") / 4).cast("int").alias("periodo"))
      .agg(F.countDistinct("household_key").alias("hogares_con_compras"),
           F.countDistinct("store_id").alias("tiendas"))
      .orderBy("periodo")
)